Elaboró:

ROJAS MARTINEZ JONATHAN FRANCISCO

# Algoritmos genéticos

In [26]:
import numpy as np
import matplotlib.pyplot as plt
import random

In [ ]:
# Valor en bots - Valor en decimal - Aptitud - probabilidad de seleccion 

In [28]:
# Función para generar un numero binario de longitud de n_bits
def num_bin_ale(n_bits):
    return [random.randint(0, 1) for _ in range(n_bits)]

In [29]:
# Función para convertir un numero binario a decimal
def decodificar_binario(bits, a, b):
    n = len(bits)
    # Convertir lista de bits a string y luego a entero decimal
    decimal = int("".join(str(bit) for bit in bits), 2)
    # Interpolar en el intervalo [a, b] (normalizar)
    return a + decimal * (b - a) / ((2**n) - 1)

In [30]:
# Funcion de aptitud
def fitness_function(x):
    return -x**3 + 60*x**2 + 15000

In [31]:
# Seleccion de los mas fuertes
def seleccion(valores, num_seleccionados):
    # ordenamos de mayor a menor la parte de aptitudes y obtenemos los indices
    indices_ordenados = valores[:, 2].argsort()[::-1]
    valores = valores[indices_ordenados]
    # seleccionamos los mejores individuos
    seleccionados = [valores[i] for i in range(num_seleccionados)]
    return seleccionados

In [139]:
# Cruza entre 2 individuos
def cruza(in1, in2, n_bits):
    # Escojemos una cantidad de puntos de cruza aleatorios
    puntos_cruza = random.randint(1, n_bits - 1) # Sin el 0 porque queremos 1 o mas puntos de cruza
    # Las posiciones empiezando de atras para adelante
    for punto in range(1, puntos_cruza + 1):
        # Intercambiamos los bits aleatoriamente entre los dos que tengan
        accion = random.choice([0, 1])
        if accion == 0:
            in1[-punto], in2[-punto] = in2[-punto], in1[-punto]
    return in1, in2


In [118]:
# Ruleta de nuevo nacimiento 
# Tenemos n individuos cada uno con una probablidad de ser seleccionado 
def ruleta_funcrandom(valores, num_seleccionados):
    # Seleccionamos un individuo basado en las probabilidades de seleccion
    seleccionados = np.random.choice(valores[:, 0], p=valores[:, 3].astype(float), size=num_seleccionados)
    return seleccionados

In [50]:
def mutacion(individuo):
    posicion = random.randrange(len(individuo))
    individuo[posicion] = 1 - individuo[posicion]  # Cambia el bit (0 a 1 o 1 a 0)
    return individuo

In [89]:
def creacion_de_generacion(poblacion, n_bits, intervalo):
    generacion = []
    # Agregamos los primeros seres vivos
    for i in range(poblacion):
        individuo = num_bin_ale(n_bits)
        valor_decimal = decodificar_binario(individuo, intervalo[0], intervalo[1])
        aptitud = fitness_function(valor_decimal)
        generacion.append([individuo, valor_decimal, aptitud])
    generacion = np.array(generacion, dtype=object)
    print("Generacion inicial:")
    print(generacion)
    return generacion

In [90]:
# Valores para el problema
n_bits = 6
intervalo = [0, 63]
poblacion = 6

In [113]:
primer_generacion = creacion_de_generacion(poblacion, n_bits, intervalo)

Generacion inicial:
[[list([1, 1, 1, 0, 1, 0]) 58.0 21728.0]
 [list([1, 1, 1, 1, 0, 1]) 61.0 11279.0]
 [list([1, 0, 0, 1, 1, 1]) 39.0 46941.0]
 [list([1, 1, 0, 0, 1, 0]) 50.0 40000.0]
 [list([0, 0, 0, 0, 0, 0]) 0.0 15000.0]
 [list([0, 1, 0, 0, 0, 0]) 16.0 26264.0]]


In [114]:
# Selección de los mejores:
def probabilidades_de_seleccion(aptitudes):
    total_aptitud = sum(aptitudes)
    probabilidades = aptitudes / total_aptitud
    return probabilidades

In [115]:
primer_generacion = np.append(primer_generacion, probabilidades_de_seleccion(primer_generacion[:, 2]).reshape(-1, 1), axis=1)

In [119]:
# Nacimiento por ruleta
primer_generacion = np.array(primer_generacion)
segunda_generacion = ruleta_funcrandom(primer_generacion, poblacion)

In [117]:
primer_generacion

array([[list([1, 1, 1, 0, 1, 0]), 58.0, 21728.0, 0.13477904870605165],
       [list([1, 1, 1, 1, 0, 1]), 61.0, 11279.0, 0.06996377440885294],
       [list([1, 0, 0, 1, 1, 1]), 39.0, 46941.0, 0.29117559486886835],
       [list([1, 1, 0, 0, 1, 0]), 50.0, 40000.0, 0.24812048730863706],
       [list([0, 0, 0, 0, 0, 0]), 0.0, 15000.0, 0.0930451827407389],
       [list([0, 1, 0, 0, 0, 0]), 16.0, 26264.0, 0.1629159119668511]],
      dtype=object)

In [127]:
def crear_gen_conbinarios(poblacion, intervalo):
    generacion = []
    for individuo in poblacion:
        valor_decimal = decodificar_binario(individuo, intervalo[0], intervalo[1])
        aptitud = fitness_function(valor_decimal)
        generacion.append([individuo, valor_decimal, aptitud])
    generacion = np.array(generacion, dtype=object)
    return generacion

In [128]:
segunda_generacion = crear_gen_conbinarios(segunda_generacion, intervalo)
print("Segunda generación:")
print(segunda_generacion)

Segunda generación:
[[list([1, 0, 0, 1, 1, 1]) 39.0 46941.0]
 [list([1, 1, 0, 0, 1, 0]) 50.0 40000.0]
 [list([0, 1, 0, 0, 0, 0]) 16.0 26264.0]
 [list([1, 0, 0, 1, 1, 1]) 39.0 46941.0]
 [list([0, 0, 0, 0, 0, 0]) 0.0 15000.0]
 [list([1, 1, 0, 0, 1, 0]) 50.0 40000.0]]


In [129]:
# Seleccion por cruza, mutacion y padres mas padres

In [130]:
num_seleccionados = 3
seleccionados = seleccion(segunda_generacion, num_seleccionados)
print("Seleccionados para cruza:")
print(seleccionados)

Seleccionados para cruza:
[array([list([1, 0, 0, 1, 1, 1]), 39.0, 46941.0], dtype=object), array([list([1, 0, 0, 1, 1, 1]), 39.0, 46941.0], dtype=object), array([list([1, 1, 0, 0, 1, 0]), 50.0, 40000.0], dtype=object)]


In [156]:
# Cruzamos entre los mejores:
tercera_generacion = []
for i in range(0, len(seleccionados)):
    for j in range(i + 1, len(seleccionados)):
        in1, in2 = seleccionados[i][0], seleccionados[j][0]
        hijo1, hijo2 = cruza(in1.copy(), in2.copy(), n_bits)
        print(f"Cruza entre {i+1} -- {in1} y {j+1} -- {in2} da como resultado: {hijo1} y {hijo2}")
        tercera_generacion.append(hijo1)
        tercera_generacion.append(hijo2)

Cruza entre 1 -- [1, 0, 0, 1, 1, 1] y 2 -- [1, 0, 0, 1, 1, 1] da como resultado: [1, 0, 0, 1, 1, 1] y [1, 0, 0, 1, 1, 1]
Cruza entre 1 -- [1, 0, 0, 1, 1, 1] y 3 -- [1, 1, 0, 0, 1, 0] da como resultado: [1, 0, 0, 1, 1, 1] y [1, 1, 0, 0, 1, 0]
Cruza entre 2 -- [1, 0, 0, 1, 1, 1] y 3 -- [1, 1, 0, 0, 1, 0] da como resultado: [1, 0, 0, 0, 1, 0] y [1, 1, 0, 1, 1, 1]


In [157]:
# Mutamos uno de la tercera generación:
individuo_a_mutar = random.choice(tercera_generacion)
print(f"Individuo antes de mutar: {individuo_a_mutar}")
individuo_mutado = mutacion(individuo_a_mutar.copy())
print(f"Individuo después de mutar: {individuo_mutado}")

Individuo antes de mutar: [1, 0, 0, 0, 1, 0]
Individuo después de mutar: [1, 1, 0, 0, 1, 0]


In [158]:
print("Tercera generación (hijos de la cruza):")
print(tercera_generacion)   

Tercera generación (hijos de la cruza):
[[1, 0, 0, 1, 1, 1], [1, 0, 0, 1, 1, 1], [1, 0, 0, 1, 1, 1], [1, 1, 0, 0, 1, 0], [1, 0, 0, 0, 1, 0], [1, 1, 0, 1, 1, 1]]


In [159]:
tercera_generacion = crear_gen_conbinarios(tercera_generacion, intervalo)
print("Tercera generación con valores decimales y aptitudes:")
print(tercera_generacion)

Tercera generación con valores decimales y aptitudes:
[[list([1, 0, 0, 1, 1, 1]) 39.0 46941.0]
 [list([1, 0, 0, 1, 1, 1]) 39.0 46941.0]
 [list([1, 0, 0, 1, 1, 1]) 39.0 46941.0]
 [list([1, 1, 0, 0, 1, 0]) 50.0 40000.0]
 [list([1, 0, 0, 0, 1, 0]) 34.0 45056.0]
 [list([1, 1, 0, 1, 1, 1]) 55.0 30125.0]]
